# SILVER_PROCEDIMENTO

Procedimentos SIGTAP com o **grupo** (2 primeiros dígitos do código, ex.: `03 - PROCEDIMENTOS CLINICOS`).

**Origem:** `BRONZE_SIGTAP` + `SILVER_DOMINIO` (GRUPO_PROCEDIMENTO).

**Estratégia de carga:** full (`TRUNCATE` + `INSERT /*+ APPEND */`), idempotente.

In [ ]:
import sys
sys.path.append("..")
from dw_utils import recarregar, consultar

In [ ]:
SQL = """
INSERT /*+ APPEND */ INTO SILVER_PROCEDIMENTO
    (COD_PROCEDIMENTO, NOME_PROCEDIMENTO, GRUPO_PROCEDIMENTO)
SELECT TRIM(s.IP_COD),
       MAX(UPPER(TRIM(s.IP_DSCR))),
       MAX(g.DESCRICAO)
FROM BRONZE_SIGTAP s
LEFT JOIN SILVER_DOMINIO g
       ON g.TABELA = 'GRUPO_PROCEDIMENTO' AND g.CODIGO = SUBSTR(TRIM(s.IP_COD), 1, 2)
WHERE TRIM(s.IP_COD) IS NOT NULL
GROUP BY TRIM(s.IP_COD)
"""

In [ ]:
recarregar("SILVER_PROCEDIMENTO", SQL, origem="BRONZE_SIGTAP")

In [ ]:
consultar("SELECT * FROM SILVER_PROCEDIMENTO FETCH FIRST 10 ROWS ONLY")